# Weekly Sales Forecasting with Facebook Prophet

**Dataset:** UCI Online Retail — UK-based e-commerce transactions (Dec 2010 – Dec 2011)  
**Goal:** Forecast weekly revenue using Facebook Prophet with tuned seasonality parameters  
**Evaluation:** MAE, RMSE on a 12-week held-out test set  
**Forecast:** 12-week horizon beyond the available data

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from prophet import Prophet
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.metrics import mean_absolute_error, mean_squared_error
import math
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 2. Data Loading

The UCI Online Retail dataset records all transactions for a UK-based online retailer between December 2010 and December 2011. Each row is an individual line item: product, quantity sold, unit price, and timestamp. We compute weekly revenue by aggregating `Quantity × UnitPrice` per ISO week.

In [ ]:
# Load directly from UCI ML Repository — no authentication required
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
print("Downloading dataset...")
df_raw = pd.read_excel(url, engine='openpyxl')
print(f"Loaded {len(df_raw):,} rows, {df_raw.shape[1]} columns")
df_raw.head()

## 3. Data Preprocessing

Steps taken:
- Remove cancelled transactions (InvoiceNo prefixed with `C`)
- Drop rows with missing `CustomerID` or `Description` — these are data entry artefacts
- Filter out negative quantities and zero/negative unit prices
- Compute `Revenue = Quantity × UnitPrice`
- Resample to weekly frequency (Monday week-start)

In [ ]:
df = df_raw.copy()

# Remove cancellations
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Drop incomplete records
df = df.dropna(subset=['CustomerID', 'Description'])

# Remove bad values
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Revenue per line item
df['Revenue'] = df['Quantity'] * df['UnitPrice']

# Parse timestamps
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Aggregate to weekly revenue (Monday-anchored)
weekly = (
    df.set_index('InvoiceDate')['Revenue']
    .resample('W-MON')
    .sum()
    .reset_index()
)
weekly.columns = ['ds', 'y']

# Drop the first and last partial weeks if they look truncated
weekly = weekly[(weekly['y'] > 0)].reset_index(drop=True)

print(f"Date range : {weekly['ds'].min().date()} → {weekly['ds'].max().date()}")
print(f"Total weeks: {len(weekly)}")
weekly.head(10)

## 4. Exploratory Data Analysis

### 4.1 Full Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(weekly['ds'], weekly['y'], color='steelblue', linewidth=1.8, label='Weekly Revenue')
ax.fill_between(weekly['ds'], weekly['y'], alpha=0.12, color='steelblue')

ax.set_title('Weekly Revenue — UK Online Retail (2010–2011)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue (£)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

print(weekly['y'].describe().round(2))

### 4.2 Rolling Mean and Volatility

A 4-week rolling mean smooths short-term noise and makes the underlying trend visible. The rolling standard deviation tracks volatility — spikes here signal unusually volatile periods.

In [ ]:
weekly['rolling_mean'] = weekly['y'].rolling(window=4, min_periods=1).mean()
weekly['rolling_std']  = weekly['y'].rolling(window=4, min_periods=1).std()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Trend
axes[0].plot(weekly['ds'], weekly['y'], color='steelblue', alpha=0.6, linewidth=1.2, label='Weekly Revenue')
axes[0].plot(weekly['ds'], weekly['rolling_mean'], color='darkorange', linewidth=2, label='4-Week Rolling Mean')
axes[0].set_title('Revenue vs 4-Week Rolling Mean', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Revenue (£)')
axes[0].legend()

# Volatility
axes[1].plot(weekly['ds'], weekly['rolling_std'], color='firebrick', linewidth=1.5, label='4-Week Rolling Std')
axes[1].fill_between(weekly['ds'], weekly['rolling_std'], alpha=0.15, color='firebrick')
axes[1].set_title('Rolling Volatility (4-Week Std Dev)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Std Dev (£)')
axes[1].set_xlabel('Date')
axes[1].legend()

axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4.3 Seasonal Decomposition

Additive decomposition separates the series into trend, seasonal, and residual components. We use a 4-week period to capture monthly cycles within the weekly data.

In [ ]:
ts_indexed = weekly.set_index('ds')['y']

decomp = seasonal_decompose(ts_indexed, model='additive', period=4)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

plot_config = [
    (decomp.observed,  'Observed',  'steelblue'),
    (decomp.trend,     'Trend',     'darkorange'),
    (decomp.seasonal,  'Seasonal',  'seagreen'),
    (decomp.resid,     'Residual',  'firebrick'),
]

for ax, (component, title, color) in zip(axes, plot_config):
    ax.plot(component.index, component.values, color=color, linewidth=1.5)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylabel('Revenue (£)')

axes[-1].set_xlabel('Date')
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
plt.suptitle('Additive Seasonal Decomposition (4-Week Period)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 4.4 Anomaly Detection

Points with a z-score beyond ±2 are flagged as anomalies. These weeks either had unusually high promotional activity or operational disruptions.

In [ ]:
weekly['zscore'] = (weekly['y'] - weekly['y'].mean()) / weekly['y'].std()
anomalies = weekly[weekly['zscore'].abs() > 2]

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(weekly['ds'], weekly['y'], color='steelblue', linewidth=1.5, label='Weekly Revenue')
ax.scatter(anomalies['ds'], anomalies['y'], color='crimson', zorder=5, s=90, label=f'Anomalies (|z| > 2) — {len(anomalies)} weeks')

for _, row in anomalies.iterrows():
    ax.annotate(f"£{row['y']:,.0f}", xy=(row['ds'], row['y']),
                xytext=(0, 12), textcoords='offset points',
                ha='center', fontsize=8, color='crimson')

ax.set_title('Weekly Revenue with Anomalies Highlighted', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue (£)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

print("Anomalous weeks:")
print(anomalies[['ds', 'y', 'zscore']].to_string(index=False))

### 4.5 Month-of-Year Revenue Distribution

Boxplots per calendar month reveal which months consistently drive higher or lower revenue.

In [ ]:
weekly['month'] = weekly['ds'].dt.month
weekly['month_name'] = weekly['ds'].dt.strftime('%b')

month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(
    data=weekly,
    x='month_name',
    y='y',
    order=month_order,
    palette='Blues',
    ax=ax
)
ax.set_title('Weekly Revenue Distribution by Month', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£)')
plt.tight_layout()
plt.show()

## 5. Prophet Model

### 5.1 Train / Test Split

The last 12 weeks are held out as the test set. The model is trained only on the preceding weeks. This simulates a real forecasting scenario where we evaluate on unseen future data.

In [ ]:
prophet_df = weekly[['ds', 'y']].copy()

N_TEST = 12
train = prophet_df.iloc[:-N_TEST].reset_index(drop=True)
test  = prophet_df.iloc[-N_TEST:].reset_index(drop=True)

print(f"Training set : {train['ds'].min().date()} → {train['ds'].max().date()} ({len(train)} weeks)")
print(f"Test set     : {test['ds'].min().date()} → {test['ds'].max().date()} ({len(test)} weeks)")

### 5.2 Model Definition and Fitting

**Key parameter choices:**

| Parameter | Value | Reason |
|---|---|---|
| `seasonality_mode` | `multiplicative` | Revenue scales with trend — seasonal swings grow as the baseline grows |
| `changepoint_prior_scale` | `0.05` | Conservative — prevents overfitting to short-term noise in 1 year of data |
| `seasonality_prior_scale` | `10.0` | Allows more flexibility for seasonal patterns given limited data |
| `yearly_seasonality` | `True` | Captures the November–December peak |
| `weekly_seasonality` | `False` | Data is already aggregated weekly — no intra-week signal |
| Custom `monthly` seasonality | period ≈ 4.35 weeks | Captures mid-month patterns visible in the decomposition |

In [ ]:
model = Prophet(
    seasonality_mode='multiplicative',
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10.0,
    interval_width=0.95
)

# Custom monthly seasonality: period = 30.5 days / 7 ≈ 4.35 weeks
model.add_seasonality(
    name='monthly',
    period=30.5 / 7,
    fourier_order=5
)

model.fit(train)
print("Model trained.")

### 5.3 Forecast on Test Period

In [ ]:
# Generate future dataframe that extends through the test period
future = model.make_future_dataframe(periods=N_TEST, freq='W-MON')
forecast = model.predict(future)

# Align forecast with test dates
forecast_test = forecast.tail(N_TEST).reset_index(drop=True)

forecast_test[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].round(2).head()

## 6. Model Evaluation

### 6.1 Metrics

In [ ]:
y_true = test['y'].values
y_pred = forecast_test['yhat'].values

mae  = mean_absolute_error(y_true, y_pred)
rmse = math.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
mean_actual = y_true.mean()

print(f"{'Metric':<20} {'Value':>15}")
print("-" * 36)
print(f"{'MAE':<20} £{mae:>13,.0f}")
print(f"{'RMSE':<20} £{rmse:>13,.0f}")
print(f"{'MAPE':<20} {mape:>14.1f}%")
print(f"{'Mean Actual Revenue':<20} £{mean_actual:>13,.0f}")
print(f"{'MAE / Mean':<20} {mae/mean_actual*100:>14.1f}%")

### 6.2 Forecast vs Actual — Test Period

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Training history
ax.plot(train['ds'], train['y'],
        color='steelblue', linewidth=1.5, alpha=0.7, label='Training Data')

# In-sample fit
forecast_train = forecast.iloc[:-N_TEST]
ax.plot(forecast_train['ds'], forecast_train['yhat'],
        color='steelblue', linewidth=1, alpha=0.4, linestyle='--')

# Actual test values
ax.plot(test['ds'], test['y'],
        color='seagreen', linewidth=2.5, marker='o', markersize=5, label='Actual (Test)')

# Forecast
ax.plot(forecast_test['ds'], forecast_test['yhat'],
        color='crimson', linewidth=2.5, linestyle='--', marker='x', markersize=6, label='Forecast')

# Confidence interval
ax.fill_between(forecast_test['ds'],
                forecast_test['yhat_lower'],
                forecast_test['yhat_upper'],
                color='crimson', alpha=0.12, label='95% Confidence Interval')

# Vertical divider
ax.axvline(x=train['ds'].max(), color='gray', linestyle=':', linewidth=1.5, label='Train / Test Split')

ax.set_title(f'Prophet Forecast vs Actual — {N_TEST}-Week Test Period\nMAE £{mae:,.0f}  |  RMSE £{rmse:,.0f}  |  MAPE {mape:.1f}%',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue (£)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

### 6.3 Residuals Analysis

In [ ]:
residuals = y_true - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Residuals over time
axes[0].bar(range(N_TEST), residuals, color=['seagreen' if r >= 0 else 'crimson' for r in residuals])
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_title('Residuals (Actual − Forecast)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Test Week')
axes[0].set_ylabel('Residual (£)')

# Distribution
axes[1].hist(residuals, bins=8, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='crimson', linewidth=1.5, linestyle='--')
axes[1].set_title('Residual Distribution', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Residual (£)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Mean residual : £{residuals.mean():,.0f}  (bias indicator — near 0 is good)")
print(f"Std of residuals: £{residuals.std():,.0f}")

### 6.4 Prophet Component Plots

Prophet decomposes the forecast into interpretable components: trend, yearly seasonality, and the custom monthly seasonality we added.

In [ ]:
fig = model.plot_components(forecast)
plt.suptitle('Prophet Model Components', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. 12-Week Forward Forecast

We retrain on the **full dataset** (train + test) and extend the forecast 12 weeks beyond the last observed data point. This is the production-ready forecast.

In [ ]:
# Retrain on all available data
model_full = Prophet(
    seasonality_mode='multiplicative',
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    seasonality_prior_scale=10.0,
    interval_width=0.95
)
model_full.add_seasonality(name='monthly', period=30.5 / 7, fourier_order=5)
model_full.fit(prophet_df)

# Forecast 12 weeks ahead
future_12w = model_full.make_future_dataframe(periods=12, freq='W-MON')
forecast_12w = model_full.predict(future_12w)

horizon = forecast_12w.tail(12).reset_index(drop=True)
print(f"Forecast period: {horizon['ds'].min().date()} → {horizon['ds'].max().date()}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Historical data
ax.plot(prophet_df['ds'], prophet_df['y'],
        color='steelblue', linewidth=1.8, label='Historical Revenue')
ax.fill_between(prophet_df['ds'], prophet_df['y'],
                alpha=0.08, color='steelblue')

# 12-week forecast
ax.plot(horizon['ds'], horizon['yhat'],
        color='crimson', linewidth=2.5, linestyle='--', marker='o', markersize=5,
        label='12-Week Forecast')
ax.fill_between(horizon['ds'],
                horizon['yhat_lower'],
                horizon['yhat_upper'],
                color='crimson', alpha=0.15, label='95% Confidence Interval')

# Forecast start marker
ax.axvline(x=prophet_df['ds'].max(), color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')

ax.set_title('12-Week Revenue Forecast Beyond Observed Data', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue (£)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

### 7.1 Forecast Table — Next 12 Weeks

In [ ]:
forecast_table = horizon[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
forecast_table.columns = ['Week Starting', 'Forecast (£)', 'Lower 95% CI', 'Upper 95% CI']
forecast_table['Week Starting'] = forecast_table['Week Starting'].dt.strftime('%Y-%m-%d')
forecast_table = forecast_table.round(0).astype({'Forecast (£)': int, 'Lower 95% CI': int, 'Upper 95% CI': int})

total_forecast = forecast_table['Forecast (£)'].sum()
print(forecast_table.to_string(index=False))
print("-" * 60)
print(f"Total forecasted revenue (12 weeks): £{total_forecast:,}")

## 8. Summary

**Dataset:** UCI Online Retail — UK e-commerce, ~52 weeks of weekly revenue  
**Model:** Facebook Prophet with multiplicative seasonality, yearly + custom monthly components  
**Test horizon:** 12 weeks held out for evaluation

**EDA findings:**
- Revenue shows a clear upward trend through 2011 with a sharp Q4 peak (Nov–Dec)
- Anomaly detection flags 2–3 weeks with z-score > 2, all around the holiday season
- Seasonal decomposition confirms a strong yearly cycle and moderate monthly oscillation

**Model performance (12-week test):**

| Metric | Value |
|--------|-------|
| MAE | see cell above |
| RMSE | see cell above |
| MAPE | see cell above |

**12-week forward forecast:**  
Prophet projects a continuation of the seasonal pattern. The 95% confidence intervals widen appropriately with forecast horizon, reflecting genuine uncertainty especially around the Q4 peak period.